# Exploration scratchpad

Interactive poking at the `human_enhancers_cohn` dataset and HyenaDNA's outputs. Nothing here is load-bearing for the pipeline in `src/` -- it's for building intuition (sequence lengths, tokenization, what the pooled representations look like).

In [ ]:
import sys
sys.path.append("../src")

from data_loading import load_enhancer_dataset, summarize_split, get_tiny_subset
from model import load_tokenizer, HyenaDNAClassifier

In [ ]:
dataset = load_enhancer_dataset()
summarize_split(dataset["train"], "train")

In [ ]:
# Sequence length distribution -- worth checking since HyenaDNA-tiny's
# context window is 1k tokens.
lengths = [len(s) for s in dataset["train"]["seq"]]
print(f"min={min(lengths)} max={max(lengths)} mean={sum(lengths)/len(lengths):.1f}")

In [ ]:
tokenizer = load_tokenizer()
sample = dataset["train"][0]["seq"]
encoded = tokenizer(sample, return_tensors="pt")
print(sample[:60], "...")
print(encoded["input_ids"][0][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0][:20]))

In [ ]:
# Sanity-check a forward pass through the frozen backbone + untrained head
# on a tiny subset -- logits should be roughly random (untrained) but the
# shapes should line up.
model = HyenaDNAClassifier()
tiny = get_tiny_subset(dataset["train"], n=5)

batch = tokenizer(tiny["seq"], padding=True, return_tensors="pt")
logits = model(batch["input_ids"], batch["attention_mask"])
print("logits shape:", logits.shape)
print("logits:", logits)
print("true labels:", tiny["label"])